In [ ]:
pip install rich

In [ ]:
import os
import time
import asyncio
from typing import Dict


def calc_mib_per_sec(total_mib: int, duration_sec: float) -> float:
    """
    Calculate throughput in mebibytes per second (MiB/s).

    :param total_mib: Total number of mebibytes written.
    :type total_mib: int
    :param duration_sec: Time taken to write in seconds.
    :type duration_sec: float
    :return: Write speed in MiB/s.
    :rtype: float
    """
    return total_mib / duration_sec if duration_sec > 0 else 0.0
    

def write_test_blocking(path: str, size_gb: int) -> None:
    """
    Write a test file of a given size (in GB) to the specified path using 100 MiB chunks.
    Reports the total write time and throughput in MiB/s. Removes the file after writing.

    :param path: Full filesystem path to the output file.
    :type path: str
    :param size_gb: Total size of the file to write, in gibibytes (GiB).
    :type size_gb: int
    :return: None
    :rtype: None

    :raises OSError: If file writing or deletion fails.
    :raises IOError: If the write operation fails due to I/O issues (e.g. disk full).
    """
    chunk_size_mb = 100
    total_size_mb = size_gb * 1024
    iterations = total_size_mb // chunk_size_mb
    chunk = b'a' * (chunk_size_mb * 1024 * 1024)

    print(f"[{path}] Writing {size_gb} GB...\n")
    start = time.time()
    try:
        with open(path, 'wb') as f:
            for _ in range(iterations):
                f.write(chunk)
        duration = time.time() - start
        speed = calc_mib_per_sec(total_size_mb, duration)
        print(f"[{path}] Done in {duration:.2f}s — {speed:.2f} MiB/s\n")
    except Exception as e:
        print(f"[{path}] ERROR during write: {e}")
        return

    try:
        os.remove(path)
        print(f"[{path}] Cleaned up.")
    except Exception as e:
        print(f"[{path}] Failed to remove file: {e}\n")

In [ ]:
import os
import time
import asyncio
from typing import Dict


def calc_mib_per_sec(total_mib: int, duration_sec: float) -> float:
    """
    Calculate throughput in mebibytes per second (MiB/s).

    :param total_mib: Total number of mebibytes written.
    :type total_mib: int
    :param duration_sec: Time taken to write in seconds.
    :type duration_sec: float
    :return: Write speed in MiB/s.
    :rtype: float
    """
    return total_mib / duration_sec if duration_sec > 0 else 0.0
    

def write_test_blocking(path: str, size_gb: int) -> None:
    """
    Write a test file of a given size (in GB) to the specified path using 100 MiB chunks.
    Reports the total write time and throughput in MiB/s. Removes the file after writing.

    :param path: Full filesystem path to the output file.
    :type path: str
    :param size_gb: Total size of the file to write, in gibibytes (GiB).
    :type size_gb: int
    :return: None
    :rtype: None

    :raises OSError: If file writing or deletion fails.
    :raises IOError: If the write operation fails due to I/O issues (e.g. disk full).
    """
    chunk_size_mb = 100
    total_size_mb = size_gb * 1024
    iterations = total_size_mb // chunk_size_mb
    chunk = b'a' * (chunk_size_mb * 1024 * 1024)

    print(f"[{path}] Writing {size_gb} GB...\n")
    start = time.time()
    try:
        with open(path, 'wb') as f:
            for _ in range(iterations):
                f.write(chunk)
        duration = time.time() - start
        speed = calc_mib_per_sec(total_size_mb, duration)
        print(f"[{path}] Done in {duration:.2f}s — {speed:.2f} MiB/s\n")
    except Exception as e:
        print(f"[{path}] ERROR during write: {e}")
        return

    try:
        #os.remove(path)
        print(f"[{path}] Cleaned up.")
    except Exception as e:
        print(f"[{path}] Failed to remove file: {e}\n")

In [ ]:
from rich.progress import (
    Progress, BarColumn, TimeElapsedColumn, TimeRemainingColumn, TaskID, ProgressColumn
)
from rich.console import Console
from rich.text import Text



task_speeds: Dict[TaskID, float] = {}


class MiBPerSecondColumn(ProgressColumn):
    """Custom column for displaying MiB/s transfer rate."""
    def render(self, task) -> Text:
        speed = task_speeds.get(task.id, None)
        return Text(f"{speed:.2f} MiB/s" if speed is not None else "-- MiB/s")
        

def write_test_with_progress(path: str, size_gb: int, progress: Progress, task_id: TaskID) -> None:
    """
    Write a test file of given size with Rich progress bar tracking.

    :param path: Full filesystem path to write to.
    :param size_gb: File size in gibibytes.
    :param progress: Rich Progress instance.
    :param task_id: Task ID returned by progress.add_task().
    """
    chunk_size_mb = 100
    total_size_mb = size_gb * 1024
    chunk = b'a' * (chunk_size_mb * 1024 * 1024)

    written = 0
    start = time.time()
    try:
        with open(path, 'wb') as f:
            while written < total_size_mb:
                f.write(chunk)
                written += chunk_size_mb
                progress.update(task_id, advance=chunk_size_mb)
                # Update MiB/s every iteration
                duration = time.time() - start
                task_speeds[task_id] = calc_mib_per_sec(written, duration)
        total_duration = time.time() - start
        progress.update(task_id, completed=total_size_mb)
        final_speed = calc_mib_per_sec(total_size_mb, total_duration)
        progress.console.print(f"\n{path} Done in {total_duration:.2f}s — {final_speed:.2f} MiB/s", markup=False)
    except Exception as e:
        progress.console.print(f"\n{path} ERROR during write: {e}", markup=False)
        return

    try:
        os.remove(path)
        progress.console.print(f"{path} Cleaned up.", markup=False)
    except Exception as e:
        progress.console.print(f"{path} Failed to remove file: {e}", markup=False)


In [ ]:
# --- non-async ---
def run_all_tests_serial(test_specs: Dict[str, int]):
    """Run the tests serially - overall performance is best in this case"""
    for path, size_gb in test_specs.items():
        write_test_blocking(path, size_gb)


# --- async/threaded ---
async def write_test_async(path: str, size_gb: int):
    """For fun, see how the system performs doing a lot of i/o async"""
    await asyncio.to_thread(write_test_blocking, path, size_gb)


async def run_all_tests_async(test_specs: Dict[str, int]):
    tasks = [write_test_async(path, size_gb) for path, size_gb in test_specs.items()]
    await asyncio.gather(*tasks)


def run_all_tests_with_rich(test_specs: Dict[str, int]) -> None:
    """
    Run all file write tests with Rich interactive progress bars.

    :param test_specs: Dictionary of file paths and GB sizes to write.
    """
    console = Console()
    with Progress(
        "[progress.description]{task.description}",
        BarColumn(),
        "[progress.percentage]{task.percentage:>3.1f}%",
        "•",
        MiBPerSecondColumn(),
        "•",
        TimeElapsedColumn(),
        "•",
        TimeRemainingColumn(),
        console=console,
    ) as progress:
        tasks = {}
        for path, size_gb in test_specs.items():
            total_mib = size_gb * 1024
            task_id = progress.add_task(f"[cyan]Writing {path}", total=total_mib)
            tasks[path] = (task_id, size_gb)

        for path, (task_id, size_gb) in tasks.items():
            write_test_with_progress(path, size_gb, progress, task_id)

In [ ]:
test_paths = {
    '/tmp/test_root_gp3.txt': 6,
    '/s3/test_s3.txt': 6,
    './test_home_gp2.txt': 6,
}

In [ ]:
# serially
run_all_tests_serial(test_paths)

In [ ]:
await run_all_tests_async(test_paths)

In [ ]:
run_all_tests_with_rich(test_paths)